# Fine-tuning & Evaluasi IndoBERT (Transformer)
**Proyek:** Analisis Sentimen Reputasi Undiksha pada Media Sosial  
**Tahap:** 5C – IndoBERT Fine-tuning (PRD §6 F6)  
**Model Pretrained:** `indobenchmark/indobert-base-p1`  


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import load_config, set_seed
from src.features.tfidf_vectorizer import split_dataset, CLASS_MAPPING
from src.features.bert_dataset import create_bert_dataloaders
from src.models.bert import IndoBERTClassifier

config = load_config('../configs/config.yaml')
set_seed(config['seed'])

df = pd.read_csv('../data/labeled/youtube_labeled.csv')
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 1. Persiapan Data & DataLoader

In [ ]:
df_train, df_val, df_test = split_dataset(
    df,
    test_size=config['split']['test_size'],
    val_size=config['split']['val_size'],
    seed=config['seed'],
)

train_loader, val_loader, test_loader, tok = create_bert_dataloaders(
    df_train, df_val, df_test,
    model_name=config['bert']['model_name'],
    max_length=config['bert']['max_length'],
    batch_size=config['bert']['batch_size'],
)
print("DataLoader siap.")


## 2. Inisialisasi Model IndoBERT & Prediksi Inferensi

In [ ]:
classifier = IndoBERTClassifier(
    model_name=config['bert']['model_name'],
    num_labels=3,
    max_length=config['bert']['max_length'],
)

sample_texts = [
    "Kampus Undiksha sangat megah dan asri dosen ramah sekali",
    "SIAKAD down terus lelet bikin pusing",
    "Pengumuman jalur mandiri gelombang dua"
]
sample_preds = classifier.predict(sample_texts)
for t, p in zip(sample_texts, sample_preds):
    print(f"Teks: {t} -> Prediksi: {p}")
